In [1]:
import pandas as pd

from transformers import (
    BartTokenizer,
    BartForConditionalGeneration
)

C:\Users\27355\Desktop\New folder\Data Science\Automated-Abstractive-Text-Summarization\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import pandas as pd

# Load the sample dataset
df = pd.read_csv("data/xsum_sample.csv")

print("✅ Dataset loaded successfully!")
print("Number of samples:", len(df))

df.head()

✅ Dataset loaded successfully!
Number of samples: 1000


,document,summary,id,document_length,summary_length
0,They recorded their best result in Wales in 30...,The general election dominated politics in Wal...,35125421,578,25
1,"More than 1,500 children were held over allege...",Children as young as 10 were among hundreds of...,35915168,415,24
2,"It includes:\nCertainly, Sinn Féin leader Gerr...",A bewildering list of developments has contrib...,39266547,635,20
3,"Then a young man in his mid-20s, the future mu...","In March 1959, as Chinese troops crushed an at...",12700331,766,21
4,"Almost 2,000 homes and more than 1,000 busines...",A £3m fund to invest in flood prevention and r...,36713610,161,18


In [5]:
print(os.listdir("data"))

['xsum_sample.csv']


In [7]:
from transformers import BartTokenizer

print("Loading BART tokenizer...")

tokenizer = BartTokenizer.from_pretrained("facebook/bart-large-cnn")

print("✅ Tokenizer loaded successfully!")

Loading BART tokenizer...
✅ Tokenizer loaded successfully!


In [9]:
from transformers import BartForConditionalGeneration

print("Loading model...")

model = BartForConditionalGeneration.from_pretrained(
    "facebook/bart-large-cnn"
)

print("Model loaded!")

Loading model...
Model loaded!


In [10]:
# Select the first article

article = df["document"][0]

print(article)

They recorded their best result in Wales in 30 years, taking 11 out of 40 seats.
Among them was Gower, which was won by just 27 seats by the former detective Byron Davies, after more than a century in the hands of a succession of Labour MPs.
Jeremy Corbyn should have some fond memories as well.
His extraordinary rise in the summer from rank outsider to Labour leader was a reflection of the party's membership in Wales, as elsewhere.
The subsequent in-fighting in the party after Corbyn's victory could prove a real problem for Welsh Labour as it looks to hold on to power at the assembly next year.
In the meantime, the current Labour Welsh government completed its final full year.
And if there was an issue which dominated in Cardiff Bay, it was once again the state of the NHS.
Opposition parties maintained the pressure on waiting lists and the decision not to protect health spending between 2011 and 2013.
Labour ministers responded by pointing out that all was not rosy in the NHS in Englan

In [11]:
inputs = tokenizer(
    article,
    max_length=512,
    truncation=True,
    return_tensors="pt"
)

print("✅ Article tokenized successfully!")

✅ Article tokenized successfully!


In [12]:
summary_ids = model.generate(
    inputs["input_ids"],
    max_length=60,
    min_length=20,
    num_beams=4,
    early_stopping=True
)

In [13]:
summary = tokenizer.decode(
    summary_ids[0],
    skip_special_tokens=True
)

print("Generated Summary")
print("=" * 60)
print(summary)

Generated Summary
Labour recorded their best result in Wales in 30 years, taking 11 out of 40 seats. Among them was Gower, which was won by just 27 seats by the former detective Byron Davies.


In [14]:
print("=" * 80)
print("ORIGINAL ARTICLE")
print("=" * 80)
print(article)

print("\n")

print("=" * 80)
print("HUMAN SUMMARY")
print("=" * 80)
print(df["summary"][0])

print("\n")

print("=" * 80)
print("BART GENERATED SUMMARY")
print("=" * 80)
print(summary)

ORIGINAL ARTICLE
They recorded their best result in Wales in 30 years, taking 11 out of 40 seats.
Among them was Gower, which was won by just 27 seats by the former detective Byron Davies, after more than a century in the hands of a succession of Labour MPs.
Jeremy Corbyn should have some fond memories as well.
His extraordinary rise in the summer from rank outsider to Labour leader was a reflection of the party's membership in Wales, as elsewhere.
The subsequent in-fighting in the party after Corbyn's victory could prove a real problem for Welsh Labour as it looks to hold on to power at the assembly next year.
In the meantime, the current Labour Welsh government completed its final full year.
And if there was an issue which dominated in Cardiff Bay, it was once again the state of the NHS.
Opposition parties maintained the pressure on waiting lists and the decision not to protect health spending between 2011 and 2013.
Labour ministers responded by pointing out that all was not rosy in 

In [15]:
results = []

for i in range(10):
    article = df["document"][i]

    inputs = tokenizer(
        article,
        max_length=512,
        truncation=True,
        return_tensors="pt"
    )

    summary_ids = model.generate(
        inputs["input_ids"],
        max_length=60,
        min_length=20,
        num_beams=4,
        early_stopping=True
    )

    generated_summary = tokenizer.decode(
        summary_ids[0],
        skip_special_tokens=True
    )

    results.append({
        "Article": article,
        "Human Summary": df["summary"][i],
        "BART Summary": generated_summary
    })

print("✅ Generated summaries for 10 articles.")

✅ Generated summaries for 10 articles.


In [16]:
results_df = pd.DataFrame(results)

results_df.to_csv(
    "data/bart_generated_summaries.csv",
    index=False
)

print("✅ Results saved successfully!")

✅ Results saved successfully!
